# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AW-OMW/FLY-RANK-PROJECT/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [2]:
from sklearn.model_selection import train_test_split
import numpy as np

# Get unique client IDs
all_client_ids = df['client_id'].unique()

# Split client IDs into training and testing sets
# We'll use a standard 80/20 split for client IDs
train_client_ids, test_client_ids = train_test_split(all_client_ids, test_size=0.2, random_state=42)

# Create training and test dataframes based on client IDs
df_train = df[df['client_id'].isin(train_client_ids)].copy()
df_test = df[df['client_id'].isin(test_client_ids)].copy()

print(f"Total unique clients: {len(all_client_ids)}")
print(f"Clients in training set: {len(train_client_ids)}")
print(f"Clients in test set: {len(test_client_ids)}")

print(f"\nShape of training data (client-based split): {df_train.shape}")
print(f"Shape of test data (client-based split): {df_test.shape}")

# Verify no client overlap
overlapping_clients = set(df_train['client_id'].unique()) & set(df_test['client_id'].unique())
if not overlapping_clients:
    print("\nSuccessfully ensured no client overlap between training and test sets.")
else:
    print(f"\nWarning: Overlapping clients found: {overlapping_clients}")

# --- DIAGNOSTIC STEP: Print columns to check for 'trend_type' ---
print("\nColumns in df_train:\n", df_train.columns)
# --- END DIAGNOSTIC STEP ---

# Re-create 'is_boom' target variable based on the newly split df_train and df_test
def create_is_boom_target(dataframe):
    # Corrected column names: 'trend_direction' instead of 'trend_type'
    # and 'trend_pct' instead of 'trend_value'
    is_positive_trend = (dataframe['trend_direction'] == 'up') & (dataframe['trend_pct'] > 0)
    is_high_search_volume = dataframe['search_volume'] > 25
    is_low_competition = dataframe['competition'] < 0.5
    is_boom = (is_positive_trend & is_high_search_volume & is_low_competition)
    return is_boom.astype(int).fillna(0)

df_train['is_boom'] = create_is_boom_target(df_train)
df_test['is_boom'] = create_is_boom_target(df_test)

# Define features (X) and target (y)
# Updated features list with corrected trend column names
features = ['search_volume', 'competition', 'trend_pct', 'trend_direction']
X_train = df_train[features]
y_train = df_train['is_boom']
X_test = df_test[features]
y_test = df_test['is_boom']

print(f"\n'is_boom' distribution in new training set:\n{y_train.value_counts()}")
print(f"'is_boom' distribution in new test set:\n{y_test.value_counts()}")

print("\nFeatures and target variables re-created for client-based split.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")


Total unique clients: 32
Clients in training set: 25
Clients in test set: 7

Shape of training data (client-based split): (26581, 44)
Shape of test data (client-based split): (3419, 44)

Successfully ensured no client overlap between training and test sets.

Columns in df_train:
 Index(['content_id', 'client_id', 'search_volume', 'competition',
       'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count',
       'char_count', 'provider_used', 'model_used', 'impressions_90d',
       'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
       'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
       'days_with_impressions', 'days_with_sessions', 'impressions_last_30d',
       'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d',
       'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier',
       'age_tier_order', 'days_since_last_update', 'freshness_tier',
       'word_count_tier', 'char_count_tier', 'ctr', 'avg_position',

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
#. 1) SEO Opportunity & Priority Queue:Predict which pages are likely to boom and rank them by priority, helping clients decide which pages to focus on first.

#2.Early Trend & Demand Detection:Identify early signals of rising search interest before the boom happens, giving clients a chance to act ahead of the trend.

#3.  Smart Content Refresh & Growth Planning:Use predictions to identify pages worth updating, expanding, or optimizing, helping clients focus resources on pages with the highest potential.

### 1. SEO Opportunity & Priority Queue: Rationale and Reason Code

**Rationale:** This action is designed to identify and prioritize content pages that have the highest potential for significant organic growth (a 'boom') in the near future. The underlying model predicts a 'boom' (where `is_boom` = 1) based on a combination of critical SEO factors: high search volume, low competition, and a positive trend in search interest. By focusing on pages that meet these criteria, clients can strategically allocate their SEO resources to areas with the greatest potential for immediate impact and high return on investment (ROI).

**How Model Predictions Support This:** The `is_boom` target variable directly indicates pages predicted to experience a surge in performance. Pages classified as `is_boom = 1` are those where the model has identified an intersection of strong demand (high search volume), manageable competition (low competition), and increasing relevance (positive `trend_pct` and `trend_direction`). This allows for a data-driven prioritization of content efforts.

**Reason Code Example:**

```
**Action: Prioritize for SEO Opportunity**
**Reason:** Our model predicts this page is likely to 'boom' due to a combination of high search volume, low competition, and an upward trend in search interest. Focusing resources here will maximize your organic growth and ROI.
```

### 2. Early Trend & Demand Detection: Rationale and Reason Code

**Rationale:** This action focuses on identifying nascent shifts in search interest and demand, allowing clients to act proactively and gain a significant competitive advantage. By detecting these early signals before they become mainstream, clients can optimize existing content, create new content, or adjust their strategies to capture emerging traffic and dominate new niches. This foresight minimizes the risk of being reactive and maximizes the potential for early market capture.

**How Model Predictions Support This:** The model continuously monitors `trend_direction` (e.g., 'up', 'down', 'stable') and `trend_pct` (the magnitude of the trend). Even if a page is not yet predicted to 'boom' (`is_boom = 0`), a consistently positive `trend_direction` and a growing `trend_pct` indicate that interest is building. The model flags these pages as potential future 'boomers' or areas of rising demand, enabling clients to invest early and secure a strong position before competition intensifies.

**Reason Code Example:**

```
**Action: Investigate Early Trend**
**Reason:** Our model detects an early upward trend in search interest and demand for this topic, indicated by a positive `trend_direction` and increasing `trend_pct`. Acting now allows you to gain a competitive edge before this trend fully matures.
```

### 3. Smart Content Refresh & Growth Planning: Rationale and Reason Code

**Rationale:** This action leverages predictive insights to guide content refresh, expansion, or optimization efforts. Instead of a blanket approach, the model enables clients to strategically allocate resources to content pages that exhibit the highest potential for improved performance, either by resolving underperformance or by capitalizing on emerging opportunities. This targeted approach ensures that content investments yield maximum impact, prevent resource wastage on low-potential pages, and maintain content relevance and authority over time.

**How Model Predictions Support This:** The model's predictions provide a granular view into each page's potential. For pages with `is_boom = 1` or strong positive `trend_direction` and `trend_pct`, a refresh might involve expanding content, updating keywords, or enhancing CTAs to capture the predicted surge in demand. For pages performing below expectations but still relevant, a refresh based on competitive analysis or updated audience insights can revive their performance. The model helps prioritize these actions by highlighting pages with favorable underlying SEO metrics (search volume, competition) that a refresh could significantly influence, thus leading to efficiency gains and measurable growth.

**Reason Code Example:**

```
**Action: Smart Content Refresh**
**Reason:** This page has a strong foundational search volume and moderate competition, but its current performance metrics indicate an opportunity for significant growth with an updated strategy. Our model suggests that a refresh, incorporating new keywords and expanded content, will likely increase its 'boom' potential or capitalize on an identified positive trend.
```

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Intended Use: To identify webpages with a high likelihood of near-term growth and prioritize them for SEO and content actions.
#Limits: Predictions are probabilistic and depend on historical data; they may be less reliable during unforeseen events, algorithm changes, or data-quality issues. The model should support, not replace, human decision-making.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#The model can prioritize opportunities, but it cannot independently approve or execute SEO changes. Predictions outside the defined confidence threshold or involving high-impact decisions require mandatory human review.

In [ ]:
#Low-confidence prediction — Below the approved confidence threshold → human review.
#Insufficient or poor-quality data — Missing, inconsistent, or too little historical data → no automated decision.
#Abnormal/temporary trends — Sudden spikes caused by events, seasonality, or external factors → human validation.
#High-impact SEO actions — Major content changes, redirects, deletions, or strategy changes → mandatory human approval.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
#Flag the model as stale when recent prediction performance falls below the validated baseline OR significant input drift is detected for a sustained period.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Reasoning**:
I need to programmatically access the content of the specified text and code cells. Since the notebook content is not directly accessible as variables in the Python environment, I will manually copy the content from the identified cells and combine them into a single string. This is the most straightforward way to gather the disparate content.



In [1]:
playbook_content = """
# 1. Ranked actions + reason codes

## 1. SEO Opportunity & Priority Queue: Rationale and Reason Code

**Rationale:** This action is designed to identify and prioritize content pages that have the highest potential for significant organic growth (a 'boom') in the near future. The underlying model predicts a 'boom' (where `is_boom` = 1) based on a combination of critical SEO factors: high search volume, low competition, and a positive trend in search interest. By focusing on pages that meet these criteria, clients can strategically allocate their SEO resources to areas with the greatest potential for immediate impact and high return on investment (ROI).

**How Model Predictions Support This:** The `is_boom` target variable directly indicates pages predicted to experience a surge in performance. Pages classified as `is_boom = 1` are those where the model has identified an intersection of strong demand (high search volume), manageable competition (low competition), and increasing relevance (positive `trend_pct` and `trend_direction`). This allows for a data-driven prioritization of content efforts.

**Reason Code Example:**

```
**Action: Prioritize for SEO Opportunity**
**Reason:** Our model predicts this page is likely to 'boom' due to a combination of high search volume, low competition, and an upward trend in search interest. Focusing resources here will maximize your organic growth and ROI.
```

## 2. Early Trend & Demand Detection: Rationale and Reason Code

**Rationale:** This action focuses on identifying nascent shifts in search interest and demand, allowing clients to act proactively and gain a significant competitive advantage. By detecting these early signals before they become mainstream, clients can optimize existing content, create new content, or adjust their strategies to capture emerging traffic and dominate new niches. This foresight minimizes the risk of being reactive and maximizes the potential for early market capture.

**How Model Predictions Support This:** The model continuously monitors `trend_direction` (e.g., 'up', 'down', 'stable') and `trend_pct' (the magnitude of the trend). Even if a page is not yet predicted to 'boom' (`is_boom = 0`), a consistently positive `trend_direction` and a growing `trend_pct` indicate that interest is building. The model flags these pages as potential future 'boomers' or areas of rising demand, enabling clients to invest early and secure a strong position before competition intensifies.

**Reason Code Example:**

```
**Action: Investigate Early Trend**
**Reason:** Our model detects an early upward trend in search interest and demand for this topic, indicated by a positive `trend_direction` and increasing `trend_pct`. Acting now allows you to gain a competitive edge before this trend fully matures.
```

## 3. Smart Content Refresh & Growth Planning: Rationale and Reason Code

**Rationale:** This action leverages predictive insights to guide content refresh, expansion, or optimization efforts. Instead of a blanket approach, the model enables clients to strategically allocate resources to content pages that exhibit the highest potential for improved performance, either by resolving underperformance or by capitalizing on emerging opportunities. This targeted approach ensures that content investments yield maximum impact, prevent resource wastage on low-potential pages, and maintain content relevance and authority over time.

**How Model Predictions Support This:** The model's predictions provide a granular view into each page's potential. For pages with `is_boom = 1` or strong positive `trend_direction` and `trend_pct`, a refresh might involve expanding content, updating keywords, or enhancing CTAs to capture the predicted surge in demand. For pages performing below expectations but still relevant, a refresh based on competitive analysis or updated audience insights can revive their performance. The model helps prioritize these actions by highlighting pages with favorable underlying SEO metrics (search volume, competition) that a refresh could significantly influence, thus leading to efficiency gains and measurable growth.

**Reason Code Example:**

```
**Action: Smart Content Refresh**
**Reason:** This page has a strong foundational search volume and moderate competition, but its current performance metrics indicate an opportunity for significant growth with an updated strategy. Our model suggests that a refresh, incorporating new keywords and expanded content, will likely increase its 'boom' potential or capitalize on an identified positive trend.
```

# 2. Intended use and limits

**Intended Use:** To identify webpages with a high likelihood of near-term growth and prioritize them for SEO and content actions.

**Limits:** Predictions are probabilistic and depend on historical data; they may be less reliable during unforeseen events, algorithm changes, or data-quality issues. The model should support, not replace, human decision-making.

# 3. Human review + the no-go list

**Human Review:** The model can prioritize opportunities, but it cannot independently approve or execute SEO changes. Predictions outside the defined confidence threshold or involving high-impact decisions require mandatory human review.

**No-Go List:**
* Low-confidence prediction — Below the approved confidence threshold → human review.
* Insufficient or poor-quality data — Missing, inconsistent, or too little historical data → no automated decision.
* Abnormal/temporary trends — Sudden spikes caused by events, seasonality, or external factors → human validation.
* High-impact SEO actions — Major content changes, redirects, deletions, or strategy changes → mandatory human approval.

# 4. Monitoring / retrain triggers

**Monitoring / Retrain Triggers:** Flag the model as stale when recent prediction performance falls below the validated baseline OR significant input drift is detected for a sustained period.
"""

print("Content consolidated into 'playbook_content' variable.")

Content consolidated into 'playbook_content' variable.


## Export Playbook to File

### Subtask:
Write the consolidated markdown content to a file named `content_action_playbook.md` within the `work/outputs/` directory. This file will serve as the export for the paper.


**Reasoning**:
The first step is to ensure the output directory exists before attempting to write the file. This prevents errors if the directory is missing.



In [2]:
import os

output_dir = "work/outputs/"
os.makedirs(output_dir, exist_ok=True)
print(f"Ensured directory '{output_dir}' exists.")

Ensured directory 'work/outputs/' exists.


**Reasoning**:
The output directory has been ensured to exist. The next step is to write the `playbook_content` string to a new file named `content_action_playbook.md` within the `work/outputs/` directory.



In [3]:
output_filepath = os.path.join(output_dir, "content_action_playbook.md")
with open(output_filepath, "w") as f:
    f.write(playbook_content)
print(f"Content action playbook successfully exported to '{output_filepath}'.")

Content action playbook successfully exported to 'work/outputs/content_action_playbook.md'.


**Reasoning**:
To confirm the successful export, I need to verify that the file actually exists at the specified path and is not empty. I will use `os.path.exists` to check for existence and `os.path.getsize` to check for content.



In [4]:
import os

output_filepath = "work/outputs/content_action_playbook.md"

if os.path.exists(output_filepath):
    if os.path.getsize(output_filepath) > 0:
        print(f"Confirmation: '{output_filepath}' exists and contains content. Export successful.")
    else:
        print(f"Warning: '{output_filepath}' exists but is empty.")
else:
    print(f"Error: '{output_filepath}' does not exist. Export failed.")

Confirmation: 'work/outputs/content_action_playbook.md' exists and contains content. Export successful.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.